In [1]:
# Cell 1 — Imports and config
import glob
import pandas as pd
from scipy import stats

DATA_DIR = 'data'

In [2]:
# Cell 2 — Load data
def load(pattern):
    files = glob.glob(f'{DATA_DIR}/{pattern}')
    assert len(files) == 1, f"Expected 1 file for {pattern}, found {len(files)}"
    return pd.read_csv(files[0])

participants    = load('participants_*.csv')
portfolio       = load('portfolio_*.csv')
task_responses  = load('task_responses_*.csv')
confidence_risk = load('confidence_risk_*.csv')
events          = load('events_*.csv')

# Ensure numeric columns are numeric
portfolio['profit_loss']             = pd.to_numeric(portfolio['profit_loss'])
task_responses['total_investment']   = pd.to_numeric(task_responses['total_investment'])
confidence_risk['confidence_rating'] = pd.to_numeric(confidence_risk['confidence_rating'])
task_responses['task_id']            = task_responses['task_id'].astype(int)
portfolio['task_id']                 = portfolio['task_id'].astype(int)
events['task_id']                    = pd.to_numeric(events['task_id'], errors='coerce')

print('Loaded:')
for name, df in [('participants', participants), ('portfolio', portfolio),
                 ('task_responses', task_responses),
                 ('confidence_risk', confidence_risk), ('events', events)]:
    print(f'  {name}: {len(df):,} rows')

Loaded:
  participants: 154 rows
  portfolio: 846 rows
  task_responses: 1,200 rows
  confidence_risk: 1,200 rows
  events: 10,808 rows


In [3]:
# Cell 3 — Sample construction

# 1. Drop withdrawn and incomplete
valid = participants[
    (participants['withdrawn'] == False) &
    (participants['completed'] == True)
].copy()
print(f'After dropping withdrawn/incomplete: {len(valid)} participants')

# 2. Attention check filtering
# Correct answers: task 3 → 2 all conditions; task 7 → 4 for e1/e6, 5 for e5
attn_rows = confidence_risk[
    confidence_risk['completed_after_task'].isin([3, 7])
][['participant_id', 'completed_after_task', 'attention_check_response']].copy()

attn_rows = attn_rows.merge(
    valid[['participant_id', 'experiment_key']], on='participant_id', how='inner'
)

def correct_answer(row):
    if row['completed_after_task'] == 3:
        return 2
    return 5 if row['experiment_key'] == 'e5' else 4

attn_rows['correct'] = attn_rows.apply(correct_answer, axis=1)
# attention_check_response is float64 (2.0, 4.0, 5.0) — compare numerically
attn_rows['passed'] = (
    pd.to_numeric(attn_rows['attention_check_response'], errors='coerce') == attn_rows['correct']
)

# Pivot to one row per participant
attn_wide = attn_rows.pivot_table(
    index='participant_id',
    columns='completed_after_task',
    values='passed',
    aggfunc='first'
).reset_index()
attn_wide.columns = ['participant_id', 'passed_task3', 'passed_task7']

# Drop only those who failed BOTH checks
failed_both = attn_wide[
    (attn_wide['passed_task3'] == False) &
    (attn_wide['passed_task7'] == False)
]['participant_id']
print(f'Dropping {len(failed_both)} participant(s) who failed both attention checks')

valid = valid[~valid['participant_id'].isin(failed_both)].copy()
print(f'Final sample: {len(valid)} participants')

After dropping withdrawn/incomplete: 110 participants
Dropping 1 participant(s) who failed both attention checks
Final sample: 109 participants


In [4]:
# Cell 4 — Sample breakdown
CONDITION_LABELS = {'e6': 'Control', 'e1': 'Treatment 1', 'e5': 'Treatment 2'}
valid['condition_label'] = valid['experiment_key'].map(CONDITION_LABELS)

print('Sample by condition:')
print(valid.groupby(['experiment_key', 'condition_label']).size().reset_index(name='n').to_string(index=False))

Sample by condition:
experiment_key condition_label  n
            e1     Treatment 1 35
            e5     Treatment 2 39
            e6         Control 35


In [5]:
# Cell 5 — Info use flags

# Exclude tutorial events: ticker in {TUT1, TUT2} OR page_name in {tutorial_1, tutorial_2}
is_tutorial = (
    events['stock_ticker'].isin(['TUT1', 'TUT2']) |
    events['page_name'].isin(['tutorial_1', 'tutorial_2'])
)
real_events = events[~is_tutorial].copy()

# Info-use event types:
#   cost_confirmation_accept + purchase-info-0  → paid info purchase (e1, e5)
#   info_request + show-more-0/show-week-0/show-month-0 → free detail views
PURCHASE_ELEMENTS  = {'purchase-info-0'}
FREE_INFO_ELEMENTS = {'show-more-0', 'show-week-0', 'show-month-0'}

info_mask = (
    ((real_events['event_type'] == 'cost_confirmation_accept') &
     (real_events['element_id'].isin(PURCHASE_ELEMENTS))) |
    ((real_events['event_type'] == 'info_request') &
     (real_events['element_id'].isin(FREE_INFO_ELEMENTS)))
)
info_events = (
    real_events[info_mask][['participant_id', 'task_id']]
    .dropna(subset=['task_id'])
    .drop_duplicates()
    .copy()
)
info_events['task_id']   = info_events['task_id'].astype(int)
info_events['info_used'] = True

print(f'Unique (participant, task) pairs with info use: {len(info_events)}')
print(f'Participants with any info use: {info_events["participant_id"].nunique()}')

Unique (participant, task) pairs with info use: 415
Participants with any info use: 63


In [6]:
# Cell 6 — Collapse to participant level

valid_ids = set(valid['participant_id'])

# Filter tables to valid participants
port_v = portfolio[portfolio['participant_id'].isin(valid_ids)].copy()
tr_v   = task_responses[task_responses['participant_id'].isin(valid_ids)].copy()
cr_v   = confidence_risk[confidence_risk['participant_id'].isin(valid_ids)].copy()

# --- avg_earnings ---
# portfolio has one row per stock per round; sum to round level first,
# then fill 0 for rounds with no investment, then average across 10 rounds
round_earnings = (
    port_v.groupby(['participant_id', 'task_id'])['profit_loss']
    .sum()
    .reset_index()
    .rename(columns={'profit_loss': 'round_profit_loss'})
)
all_rounds = tr_v[['participant_id', 'task_id']].merge(
    round_earnings, on=['participant_id', 'task_id'], how='left'
)
all_rounds['round_profit_loss'] = all_rounds['round_profit_loss'].fillna(0)
avg_earnings = (
    all_rounds.groupby('participant_id')['round_profit_loss']
    .mean()
    .reset_index()
    .rename(columns={'round_profit_loss': 'avg_earnings'})
)

# --- investment_rate ---
tr_v['invested'] = tr_v['total_investment'] > 0
investment_rate = (
    tr_v.groupby('participant_id')['invested']
    .mean()
    .reset_index()
    .rename(columns={'invested': 'investment_rate'})
)

# --- info_use_rate ---
tr_with_info = tr_v[['participant_id', 'task_id']].merge(
    info_events, on=['participant_id', 'task_id'], how='left'
)
tr_with_info['info_used'] = tr_with_info['info_used'].fillna(False)
info_use_rate = (
    tr_with_info.groupby('participant_id')['info_used']
    .mean()
    .reset_index()
    .rename(columns={'info_used': 'info_use_rate'})
)

# --- avg_confidence ---
avg_confidence = (
    cr_v.groupby('participant_id')['confidence_rating']
    .mean()
    .reset_index()
    .rename(columns={'confidence_rating': 'avg_confidence'})
)

# --- Merge all into one participant-level DataFrame ---
participant_df = (
    valid[['participant_id', 'experiment_key', 'condition_label']]
    .merge(avg_earnings,    on='participant_id', how='left')
    .merge(investment_rate, on='participant_id', how='left')
    .merge(info_use_rate,   on='participant_id', how='left')
    .merge(avg_confidence,  on='participant_id', how='left')
)

print(f'Shape: {participant_df.shape}  (expected: {len(valid)} rows x 7 cols)')
print(f'\nMissing values:\n{participant_df.isnull().sum()}')

Shape: (109, 7)  (expected: 109 rows x 7 cols)

Missing values:
participant_id     0
experiment_key     0
condition_label    0
avg_earnings       0
investment_rate    0
info_use_rate      0
avg_confidence     0
dtype: int64


In [7]:
# Cell 7 — Sanity checks
OUTCOMES = ['avg_earnings', 'investment_rate', 'info_use_rate', 'avg_confidence']

print('Outcome ranges:')
print(participant_df[OUTCOMES].describe().round(3))

print('\nSanity checks:')
assert participant_df['investment_rate'].between(0, 1).all(), 'investment_rate out of [0,1]'
assert participant_df['info_use_rate'].between(0, 1).all(),   'info_use_rate out of [0,1]'
assert participant_df['avg_confidence'].between(0, 100).all(), 'avg_confidence out of [0,100]'
print('  investment_rate in [0,1]: OK')
print('  info_use_rate in [0,1]:   OK')
print('  avg_confidence in [0,100]: OK')

Outcome ranges:
       avg_earnings  investment_rate  avg_confidence
count       109.000          109.000         109.000
mean          0.156            0.705          57.270
std           3.726            0.227          21.378
min          -9.417            0.100           0.000
25%          -2.033            0.500          46.300
50%          -0.041            0.700          60.000
75%           1.208            0.900          72.300
max          14.870            1.000         100.000

Sanity checks:
  investment_rate in [0,1]: OK
  info_use_rate in [0,1]:   OK
  avg_confidence in [0,100]: OK


In [8]:
# Cell 8 — One-way ANOVAs
OUTCOMES = ['avg_earnings', 'investment_rate', 'info_use_rate', 'avg_confidence']
conditions = sorted(participant_df['experiment_key'].unique())

print('=' * 65)
print('ONE-WAY ANOVA: condition (e1, e5, e6) as independent variable')
print('=' * 65)

for outcome in OUTCOMES:
    groups = [
        participant_df.loc[participant_df['experiment_key'] == c, outcome].dropna().values
        for c in conditions
    ]
    f_stat, p_val = stats.f_oneway(*groups)
    sig = '* SIGNIFICANT (p < 0.05)' if p_val < 0.05 else 'not significant'

    print(f'\nOutcome: {outcome}')
    print(f'  F = {f_stat:.3f},  p = {p_val:.4f}  [{sig}]')
    print('  Condition means:')
    means = (
        participant_df
        .groupby(['experiment_key', 'condition_label'])[outcome]
        .mean()
        .round(3)
    )
    for (exp_key, label), mean_val in means.items():
        n = (participant_df['experiment_key'] == exp_key).sum()
        print(f'    {exp_key} ({label}, n={n}): {mean_val}')

print('\n' + '=' * 65)

ONE-WAY ANOVA: condition (e1, e5, e6) as independent variable

Outcome: avg_earnings
  F = 4.019,  p = 0.0208  [* SIGNIFICANT (p < 0.05)]
  Condition means:
    e1 (Treatment 1, n=35): 0.447
    e5 (Treatment 2, n=39): -1.091
    e6 (Control, n=35): 1.253

Outcome: investment_rate
  F = 0.346,  p = 0.7083  [not significant]
  Condition means:
    e1 (Treatment 1, n=35): 0.726
    e5 (Treatment 2, n=39): 0.682
    e6 (Control, n=35): 0.709

Outcome: info_use_rate
  F = 34.208,  p = 0.0000  [* SIGNIFICANT (p < 0.05)]
  Condition means:
    e1 (Treatment 1, n=35): 0.434
    e5 (Treatment 2, n=39): 0.623
    e6 (Control, n=35): 0.0

Outcome: avg_confidence
  F = 2.064,  p = 0.1321  [not significant]
  Condition means:
    e1 (Treatment 1, n=35): 53.169
    e5 (Treatment 2, n=39): 62.662
    e6 (Control, n=35): 55.363

